In [4]:
# =========================
# LIBRARIES
# =========================
import pandas as pd
import numpy as np
import re

from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

print("Libraries loaded")

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("data/processed/bank_reviews_clean.csv")
df = df.dropna(subset=["review"])

print("Data loaded:", df.shape)

# =========================
# SENTIMENT MODEL (DISTILBERT)
# =========================
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# limit for stability (remove if strong machine)
texts = df["review"].astype(str).tolist()

print("Running sentiment...")

results = sentiment_model(texts[:2000])

df = df.iloc[:len(results)].copy()
df["sentiment_label"] = [r["label"].lower() for r in results]
df["sentiment_score"] = [r["score"] for r in results]

print("Sentiment done")

# =========================
# CLEAN TEXT
# =========================
def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text

df["clean"] = df["review"].apply(clean)

# =========================
# TF-IDF THEME EXTRACTION
# =========================
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2), max_features=800)
X = vectorizer.fit_transform(df["clean"])
terms = vectorizer.get_feature_names_out()

themes_by_bank = {}

for bank in df["bank"].unique():
    sub = df[df["bank"] == bank]
    sub_vec = vectorizer.transform(sub["clean"])

    scores = np.asarray(sub_vec.sum(axis=0)).flatten()
    top_idx = scores.argsort()[-10:][::-1]

    themes_by_bank[bank] = [terms[i] for i in top_idx]

print("\nTop keywords per bank:")
print(themes_by_bank)

# =========================
# THEME CLASSIFICATION
# =========================
def assign_theme(text):
    text = str(text)

    if any(x in text for x in ["login", "otp", "password", "sign"]):
        return "Account Access Issues"

    if any(x in text for x in ["transfer", "transaction", "slow", "delay"]):
        return "Transaction Performance"

    if any(x in text for x in ["ui", "design", "interface"]):
        return "UI & Experience"

    if any(x in text for x in ["error", "bug", "crash", "fail"]):
        return "App Stability"

    if any(x in text for x in ["feature", "request", "add", "update"]):
        return "Feature Requests"

    return "Other"

df["theme"] = df["clean"].apply(assign_theme)

# =========================
# INSIGHTS
# =========================
print("\n=== SENTIMENT BY BANK ===")
print(df.groupby("bank")["sentiment_score"].mean())

print("\n=== THEMES BY BANK ===")
print(df.groupby(["bank", "theme"]).size())

# =========================
# RATING VS SENTIMENT CHECK
# =========================
print("\n=== MISMATCH ANALYSIS ===")

mismatch = df[
    ((df["rating"] >= 4) & (df["sentiment_label"] == "negative")) |
    ((df["rating"] <= 2) & (df["sentiment_label"] == "positive"))
]

print("Mismatch count:", len(mismatch))

# =========================
# FINAL OUTPUT
# =========================
final_df = df[[
    "review",
    "rating",
    "date",
    "bank",
    "sentiment_label",
    "sentiment_score",
    "theme"
]]

final_df.to_csv("data/processed/task2_final_output.csv", index=False)

print("\nSaved ✔ task2_final_output.csv")

print("\nDONE ✔ Task 2 complete")

Libraries loaded
Data loaded: (1200, 5)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Running sentiment...
Sentiment done

Top keywords per bank:
{'CBE': ['good', 'app', 'nice', 'best', 'ok', 'cbe', 'like', 'nice app', 'application', 'excellent'], 'Dashen': ['good', 'app', 'nice', 'best', 'bank', 'dashen', 'super', 'great', 'ok', 'fast']}

=== SENTIMENT BY BANK ===
bank
CBE       0.975562
Dashen    0.974892
Name: sentiment_score, dtype: float64

=== THEMES BY BANK ===
bank    theme                  
CBE     Account Access Issues        2
        App Stability               12
        Feature Requests            25
        Other                      515
        Transaction Performance     40
        UI & Experience              6
Dashen  Account Access Issues       20
        App Stability               10
        Feature Requests            22
        Other                      491
        Transaction Performance     40
        UI & Experience             17
dtype: int64

=== MISMATCH ANALYSIS ===
Mismatch count: 197

Saved ✔ task2_final_output.csv

DONE ✔ Task 2 comple